In [ ]:
import os
import torch
import timm
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.amp import autocast, GradScaler

c:\Users\preeti\Desktop\CODERAY\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. SETUP
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 50 
CHECKPOINT_INTERVAL = 10
# Replace with your actual path if it changes
DATA_PATH = r"C:\Users\preeti\Desktop\x-rayData\Xray\Bone -Fracture\Bone Fracture\Augmented\Comminuted Bone Fracture"

print(f"Data Path set to: {DATA_PATH}")

In [ ]:
# 2. DATASET (Strictly Unlabeled)
class UnlabeledXrayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        for root, _, files in os.walk(root_dir):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(root, f))
        self.transform = transform
        print(f"Found {len(self.image_paths)} images.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img
        except Exception as e:
            return torch.zeros((3, 224, 224)) # Fallback for corrupt files

In [ ]:
# Define Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), 
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Initialize Loader
dataset = UnlabeledXrayDataset(DATA_PATH, transform=transform)
loader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0, # Set to 0 in Jupyter/Windows to avoid multiprocessing issues
    pin_memory=True
)
print("Initializing MAE Model...")

In [ ]:
# IMPORTANT: Using the specific MAE version of the model
model = timm.create_model('vit_large_patch16_224_mae', pretrained=False, mask_ratio=0.75).to(DEVICE)
# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler(device_type='cuda')
print("Model ready for training.")

In [ ]:
# --- Early Stopping Config ---
patience = 5          # How many epochs to wait for improvement
min_delta = 0.00001   # Minimum change to qualify as an improvement
best_loss = float('inf')
counter = 0

loss_history = []
model.train()

print("Starting effective pre-training with Early Stopping...")

for epoch in range(EPOCHS):
    total_loss = 0
    for batch_idx, images in enumerate(loader):
        images = images.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', dtype=torch.float16, enabled=torch.cuda.is_available()):
            output = model(images)
            loss = output[0] if isinstance(output, (tuple, list)) else output

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {avg_loss:.6f}")

    # --- EARLY STOPPING LOGIC ---
    if avg_loss < (best_loss - min_delta):
        best_loss = avg_loss
        counter = 0  # Reset counter because we found a new best!
        # Save the best version of the weights so far
        torch.save(model.state_dict(), "mae_bone_features_best.pth")
    else:
        counter += 1
        print(f"--- No improvement for {counter} epoch(s). ---")

    if counter >= patience:
        print(f"Early stopping triggered at Epoch {epoch+1}. Model is fully trained.")
        break

# Final Save
torch.save(model.state_dict(), "mae_bone_features_final.pth")

In [ ]:
import numpy as np

def visualize_reconstruction(model, loader, device, n_images=2):
    model.eval()
    with torch.no_grad():
        # 1. Get a single batch of real X-rays
        images = next(iter(loader))
        images = images.to(device)
        
        # 2. Forward pass through MAE
        # Returns: Loss (scalar), Prediction (patches), Mask (binary)
        loss, pred, mask = model(images)
        
        # 3. Unpatchify: Stitch the predicted patches back into a 224x224 image
        pred_img = model.unpatchify(pred)
        
        # 4. Prepare data for plotting
        images = images.cpu()
        pred_img = pred_img.cpu()
        mask = mask.cpu()
        
        # 5. Create the "Masked" image to show what the model was given
        # We expand the mask to match the pixel dimensions
        p = model.patch_embed.patch_size[0]
        mask = mask.detach()
        mask = mask.unsqueeze(-1).repeat(1, 1, p**2 * 3)
        mask = model.unpatchify(mask)
        im_masked = images * (1 - mask)

        # 6. PLOTTING
        plt.figure(figsize=(15, 5 * n_images))
        for i in range(n_images):
            # Column 1: Original
            plt.subplot(n_images, 3, i*3 + 1)
            plt.imshow(images[i].permute(1, 2, 0).numpy().clip(0, 1))
            plt.title("Original X-ray")
            plt.axis('off')

            # Column 2: The Masked Input
            plt.subplot(n_images, 3, i*3 + 2)
            plt.imshow(im_masked[i].permute(1, 2, 0).numpy().clip(0, 1))
            plt.title("What Model Sees (75% Masked)")
            plt.axis('off')

            # Column 3: The Model's Reconstruction
            plt.subplot(n_images, 3, i*3 + 3)
            plt.imshow(pred_img[i].permute(1, 2, 0).numpy().clip(0, 1))
            plt.title("MAE Reconstruction")
            plt.axis('off')
        
        plt.tight_layout()
        plt.show()

# Execute the visualization
visualize_reconstruction(model, loader, DEVICE)

In [ ]:
import torch.nn as nn

class XrayFractureClassifier(nn.Module):
    def __init__(self, checkpoint_path=None, num_classes=2):
        super().__init__()
        # Load the base architecture
        self.encoder = timm.create_model('vit_base_patch16_224.mae', pretrained=False)
        
        # Load your pretrained MAE weights
        if checkpoint_path:
            print(f"Loading pretrained weights from {checkpoint_path}...")
            # We use strict=False because the MAE checkpoint has a 'decoder' 
            # which we are intentionally throwing away here.
            state_dict = torch.load(checkpoint_path, map_location="cpu")
            self.encoder.load_state_dict(state_dict, strict=False)
        
        # With 40k images, we want the encoder to learn!
        for param in self.encoder.parameters():
            param.requires_grad = True
        
        # Final classification head
        self.head = nn.Linear(self.encoder.num_features, num_classes)

    def forward(self, x):
        # Extract features from the ViT encoder
        x = self.encoder.forward_features(x)
        # Global average pooling (take the mean of all tokens)
        x = x.mean(dim=1) 
        return self.head(x)

# Initialize for your large dataset
# model = XrayFractureClassifier("mae_bone_features_final.pth", num_classes=2).to(DEVICE)

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

# 1. Define Supervised Transforms (slightly more aggressive for classification)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), # Bones can be oriented differently
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load Labeled Data (Assumes folders: /train/Fracture and /train/Normal)
# Replace with your labeled data path
LABELED_DATA_PATH = r"C:\Users\preeti\Desktop\x-rayData\Labeled"
full_dataset = ImageFolder(root=LABELED_DATA_PATH)

# 3. Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# Apply specific transforms to each subset
train_subset.dataset.transform = train_transform
val_subset.dataset.transform = val_transform

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Training on: {train_size} images | Validating on: {val_size} images")

In [ ]:
import torch.nn.functional as F

# Initialize Model (loading your best MAE weights)
model = XrayFractureClassifier("mae_bone_features_final.pth", num_classes=2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5) # Lower LR for fine-tuning
criterion = nn.CrossEntropyLoss()

for epoch in range(10): # Fine-tuning usually takes fewer epochs
    # --- TRAINING PHASE ---
    model.train()
    train_loss, correct = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()
            
    print(f"Epoch {epoch+1}: Train Acc: {100*correct/train_size:.2f}% | Val Acc: {100*val_correct/val_size:.2f}%")

In [ ]:
import matplotlib.pyplot as plt 

def plot_pretraining_results(losses):
    plt.figure(figsize=(10, 5))
    plt.plot(losses, label='Reconstruction Loss', color='#2ca02c', linewidth=2)
    plt.title('MAE Pretraining Progress (Bone Features)')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

# To use this, add 'loss_history = []' before your training loop 
# and 'loss_history.append(avg_loss)' inside the epoch loop.
# Then call:
# plot_pretraining_results(loss_history)